In [0]:
/*
  Script: Load Silver
  Description:
    This script loads and cleans data from bronze layer into silver layer.
    It standardizes formats, removes invalid records, deduplicates data, and ensures data quality across CRM and ERP tables.
*/

CREATE OR REPLACE PROCEDURE load_silver()
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    SELECT '=================================================';
    SELECT 'Loading silver layer...';
    SELECT '=================================================';

    SELECT '-------------------------------------------------';
    SELECT 'Loading CRM tables...';
    SELECT '-------------------------------------------------';

    /*
    ================================================================
    Table: crm_cust_info
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.crm_cust_info...';

    TRUNCATE TABLE silver.crm_cust_info;

    -- Remove null ids
    -- Remove null history rows
    -- Remove unwanted spaces in string columns
    -- Standardize column values in cst_gndr
    -- Standardize column values in cst_marital_status
    INSERT INTO silver.crm_cust_info (
        cst_id,
        cst_key,
        cst_firstname,
        cst_lastname,
        cst_marital_status,
        cst_gndr,
        cst_create_date
    )
    (
        SELECT
            cst_id,
            cst_key,
            TRIM(cst_firstname) AS cst_firstname,
            TRIM(cst_lastname) AS cst_lastname,
            CASE UPPER(TRIM(cst_marital_status))
                WHEN 'S' THEN 'Single'
                WHEN 'M' THEN 'Married'
                ELSE 'n/a'
            END AS cst_marital_status,
            CASE UPPER(TRIM(cst_gndr))
                WHEN 'F' THEN 'Female'
                WHEN 'M' THEN 'Male'
                ELSE 'n/a'
            END AS cst_gndr,
            cst_create_date
        FROM (
            SELECT
                *,
                RANK() OVER (
                    PARTITION BY cst_key, cst_id
                    ORDER BY cst_create_date DESC
                ) AS ver_rank
            FROM bronze.crm_cust_info
            WHERE cst_id IS NOT NULL
        ) t
        WHERE ver_rank = 1
        ORDER BY cst_create_date DESC
    );

    /*
    ================================================================
    Table: crm_prd_info
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.crm_prd_info...';

    TRUNCATE TABLE silver.crm_prd_info;

    -- Split into new cat_id and standardized prd_key columns
    -- Standardize column values in prd_line
    -- Set default prd_cost to 0
    -- Update prd_end_dt to be based on next prd_start_dt instance of a product
    INSERT INTO silver.crm_prd_info (
        prd_id,
        cat_id,
        prd_key,
        prd_nm,
        prd_cost,
        prd_line,
        prd_start_dt,
        prd_end_dt
    )
    (
        SELECT
            prd_id,
            REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
            SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,
            prd_nm,
            COALESCE(prd_cost, 0) AS prd_cost,
            CASE UPPER(TRIM(prd_line))
                WHEN 'M' THEN 'Mountain'
                WHEN 'R' THEN 'Road'
                WHEN 'S' THEN 'Other Sales'
                WHEN 'T' THEN 'Touring'
                ELSE 'n/a'
            END AS prd_line,
            CAST(prd_start_dt AS DATE) AS prd_start_dt,
            CAST(
                DATEADD(day, -1, LEAD(prd_start_dt) OVER (
                    PARTITION BY prd_key
                    ORDER BY prd_start_dt ASC
                )) AS DATE
            ) AS prd_end_dt
        FROM bronze.crm_prd_info
    );

    /*
    ================================================================
    Table: crm_sales_details
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.crm_sales_details...';

    TRUNCATE TABLE silver.crm_sales_details;

    -- Convert sls_order_dt, sls_ship_dt and sls_due_dt to DATE and handle invalid values
    -- Calculate and rewrite invalid sls_price, sls_ sales and sls_quantity values
    INSERT INTO silver.crm_sales_details (
        sls_ord_num,
        sls_prd_key,
        sls_cust_id,
        sls_order_dt,
        sls_ship_dt,
        sls_due_dt,
        sls_sales,
        sls_quantity,
        sls_price
    )
    SELECT
        sls_ord_num,
        sls_prd_key,
        sls_cust_id,

        CASE
            WHEN sls_order_dt = 0 OR LENGTH(sls_order_dt) != 8 THEN NULL
            ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
        END,

        CASE
            WHEN sls_ship_dt = 0 OR LENGTH(sls_ship_dt) != 8 THEN NULL
            ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd')
        END,

        CASE
            WHEN sls_due_dt = 0 OR LENGTH(sls_due_dt) != 8 THEN NULL
            ELSE TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd')
        END,

        CASE
            WHEN sls_sales != sls_quantity * (
                CASE
                    WHEN sls_price IS NULL OR sls_price <= 0
                    THEN sls_sales / NULLIF(sls_quantity, 0)
                    ELSE ABS(sls_price)
                END
            )
            AND sls_quantity = sls_sales * (
                CASE
                    WHEN sls_price IS NULL OR sls_price <= 0
                    THEN sls_sales / NULLIF(sls_quantity, 0)
                    ELSE ABS(sls_price)
                END
            )
            THEN sls_quantity

            WHEN sls_sales IS NULL
                OR sls_sales <= 0
                OR sls_sales != sls_quantity * (
                    CASE
                        WHEN sls_price IS NULL OR sls_price <= 0
                        THEN sls_sales / NULLIF(sls_quantity, 0)
                        ELSE ABS(sls_price)
                    END
                )
            THEN sls_quantity * (
                CASE
                    WHEN sls_price IS NULL OR sls_price <= 0
                    THEN sls_sales / NULLIF(sls_quantity, 0)
                    ELSE ABS(sls_price)
                END
            )

            ELSE sls_sales
        END,

        CASE
            WHEN sls_sales != sls_quantity * (
                CASE
                    WHEN sls_price IS NULL OR sls_price <= 0
                    THEN sls_sales / NULLIF(sls_quantity, 0)
                    ELSE ABS(sls_price)
                END
            )
            AND sls_quantity = sls_sales * (
                CASE
                    WHEN sls_price IS NULL OR sls_price <= 0
                    THEN sls_sales / NULLIF(sls_quantity, 0)
                    ELSE ABS(sls_price)
                END
            )
            THEN sls_sales
            ELSE sls_quantity
        END,

        CASE
            WHEN sls_price IS NULL OR sls_price <= 0
            THEN sls_sales / NULLIF(sls_quantity, 0)
            ELSE ABS(sls_price)
        END

    FROM bronze.crm_sales_details;

    SELECT '-------------------------------------------------';
    SELECT 'Loading ERP tables...';
    SELECT '-------------------------------------------------';

    /*
    ================================================================
    Table: erp_cust_az12
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.erp_cust_az12...';

    TRUNCATE TABLE silver.erp_cust_az12;

    -- Standardize cid and gen values
    -- Remove invalid bdate values
    INSERT INTO silver.erp_cust_az12 (
        cid,
        bdate,
        gen
    )
    (
        SELECT
            CASE
                WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4)
                ELSE cid
            END AS cid,
            CASE
                WHEN bdate > GETDATE() THEN NULL
                ELSE bdate
            END AS bdate,
            CASE
                WHEN TRIM(gen) IS NULL OR TRIM(gen) = '' THEN 'n/a'
                WHEN TRIM(gen) = 'F' THEN 'Female'
                WHEN TRIM(gen) = 'M' THEN 'Male'
                ELSE TRIM(gen)
            END AS gen
        FROM bronze.erp_cust_az12
    );

    /*
    ================================================================
    Table: erp_loc_a101
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.erp_loc_a101...';

    TRUNCATE TABLE silver.erp_loc_a101;

    -- Standardize cid and cntry values
    INSERT INTO silver.erp_loc_a101 (
        cid,
        cntry
    )
    (
        SELECT
            REPLACE(cid, '-', '') AS cid,
            CASE
                WHEN cntry = NULL OR TRIM(cntry) = '' THEN 'n/a'
                WHEN TRIM(cntry) = 'DE' THEN 'GERMANY'
                WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
                ELSE TRIM(cntry)
            END AS cntry
        FROM bronze.erp_loc_a101
    );

    /*
    ================================================================
    Table: erp_px_cat_g1v2
    ================================================================
    */

    SELECT '>> Truncating and adding new rows to: silver.erp_px_cat_g1v2...';

    TRUNCATE TABLE silver.erp_px_cat_g1v2;

    -- Insert data as-is (no issues)
    INSERT INTO silver.erp_px_cat_g1v2 (
        id,
        cat,
        subcat,
        maintenance
    )
    (
        SELECT
            id,
            cat,
            subcat,
            maintenance
        FROM bronze.erp_px_cat_g1v2
    );

    -- Update missing values from downstream tables
    INSERT INTO silver.erp_px_cat_g1v2 (
        id,
        cat,
        subcat,
        maintenance
    )
    SELECT DISTINCT
        cat_id AS id,
        NULL AS cat,
        NULL AS subcat,
        NULL AS maintenance
    FROM silver.crm_prd_info p
    WHERE NOT EXISTS (
        SELECT 1
        FROM silver.erp_px_cat_g1v2 t
        WHERE t.id = p.cat_id
    );

    SELECT 'SUCCESS. All tasks performed successfully.';
END;

-- Example
-- CALL load_silver();